PRE REQ TO RUNNING THIS:
- Must update filepath with new downloaded data file (manually move it from downloads) 
- Must update output_file with year and month of release

In [108]:
import pandas as pd
%pip install geopandas
import geopandas as gpd
import numpy as np
import time
import subprocess
from shapely.geometry import MultiPolygon
import pickle
import os


from datetime import date
from os import path
from shapely.geometry import LineString, Point
from IPython.display import display
# %pip install xlswriter
# import xlsxwriter
%pip install openpyxl
import openpyxl
from openpyxl import Workbook
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.styles import Alignment
from datetime import datetime, timedelta 




[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [109]:
today_date = datetime.today()

iso_today_date = today_date.isoformat().split('T')[0]

In [110]:
full_country_list = [
    "Algeria", "Angola", "Benin", "Botswana", "British Indian Ocean Territory", "Burkina Faso", 
    "Burundi", "Cabo Verde", "Cameroon", "Central African Republic", "Chad", "Comoros", "DR Congo", 
    "Republic of the Congo", "Côte d'Ivoire", "Djibouti", "Egypt", "Equatorial Guinea", "Eritrea", 
    "Eswatini", "Ethiopia", "French Southern Territories", "Gabon", "The Gambia", "Ghana", "Guinea", 
    "Guinea-Bissau", "Kenya", "Lesotho", "Liberia", "Libya", "Madagascar", "Malawi", "Mali", "Mauritania", 
    "Mauritius", "Mayotte", "Morocco", "Mozambique", "Namibia", "Niger", "Nigeria", "Réunion", "Rwanda", 
    "Saint Helena, Ascension, and Tristan da Cunha", "Sao Tome and Principe", "Senegal", "Seychelles", 
    "Sierra Leone", "Somalia", "South Africa", "South Sudan", "Sudan", "Tanzania", "Togo", "Tunisia", 
    "Uganda", "Western Sahara", "Zambia", "Zimbabwe",
    
    "Anguilla", "Antigua and Barbuda", "Argentina", "Aruba", "Bahamas", "Barbados", "Belize", "Bermuda", 
    "Bolivia", "Bonaire, Sint Eustatius, and Saba", "Bouvet Island", "Brazil", "Canada", "Cayman Islands", 
    "Chile", "Colombia", "Costa Rica", "Cuba", "Curaçao", "Dominica", "Dominican Republic", "Ecuador", 
    "El Salvador", "Falkland Islands", "French Guiana", "Greenland", "Grenada", "Guadeloupe", "Guatemala", 
    "Guyana", "Haiti", "Honduras", "Jamaica", "Martinique", "Mexico", "Montserrat", "Nicaragua", "Panama", 
    "Paraguay", "Peru", "Puerto Rico", "Saint Barthélemy", "Saint Kitts and Nevis", "Saint Lucia", 
    "Saint Martin (French part)", "Saint Pierre and Miquelon", "Saint Vincent and the Grenadines", 
    "Sint Maarten (Dutch part)", "South Georgia and the South Sandwich Islands", "Suriname", 
    "Trinidad and Tobago", "Turks and Caicos Islands", "United States", "Uruguay", "Venezuela", 
    "Virgin Islands (British)", "Virgin Islands (U.S.)",
    
    "Afghanistan", "Armenia", "Azerbaijan", "Bahrain", "Bangladesh", "Bhutan", "Brunei", "Cambodia", "China", 
    "Cyprus", "Georgia", "Hong Kong", "India", "Indonesia", "Iran", "Iraq", "Israel", "Japan", "Jordan", 
    "Kazakhstan", "North Korea", "South Korea", "Kuwait", "Kyrgyzstan", "Laos", "Lebanon", "Macao", 
    "Malaysia", "Maldives", "Mongolia", "Myanmar", "Nepal", "Oman", "Pakistan", "Palestine", "Philippines", 
    "Qatar", "Saudi Arabia", "Singapore", "Sri Lanka", "Syria", "Taiwan", "Tajikistan", "Thailand", 
    "Timor-Leste", "Türkiye", "Turkmenistan", "United Arab Emirates", "Uzbekistan", "Vietnam", "Yemen",
    
    "Åland Islands", "Albania", "Andorra", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina", 
    "Bulgaria", "Croatia", "Czech Republic", "Denmark", "Estonia", "Faroe Islands", "Finland", "France", 
    "Germany", "Gibraltar", "Greece", "Guernsey", "Holy See", "Hungary", "Iceland", "Ireland", "Isle of Man", 
    "Italy", "Jersey", "Kosovo", "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "North Macedonia", 
    "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands", "Norway", "Poland", "Portugal", "Romania", 
    "Russia", "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", "Svalbard and Jan Mayen", "Sweden", 
    "Switzerland", "Ukraine", "United Kingdom",
    
    "American Samoa", "Australia", "Christmas Island", "Cocos (Keeling) Islands", "Cook Islands", "Fiji", 
    "French Polynesia", "Guam", "Heard Island and McDonald Islands", "Kiribati", "Marshall Islands", 
    "Micronesia", "Nauru", "New Caledonia", "New Zealand", "Niue", "Norfolk Island", 
    "Northern Mariana Islands", "Palau", "Papua New Guinea", "Pitcairn", "Samoa", "Solomon Islands", 
    "Tokelau", "Tonga", "Tuvalu", "United States Minor Outlying Islands", "Vanuatu", "Wallis and Futuna"
]

In [111]:
# changed to old_data just to test after moving gcpft.ipynb to the coal finance folder before it was just in the green info maps folder
filepath = 'old_data/Global Coal Project Finance Tracker_November 2025_Global Energy Monitor DATA TEAM COPY.xlsx'
# filepath = '/Users/gem-tah/GEM_INFO/GEM_WORK/greeninfo-maps/coal-finance-main/old_data/Global Coal Project Finance Tracker_November 2025_Global Energy Monitor DATA TEAM COPY.xlsx'
# filepath = '/Users/gem-tah/GEM_INFO/GEM_WORK/greeninfo-maps/coal-finance-main/data/Global Coal Project Finance Tracker_November 2025_Global Energy Monitor DATA TEAM COPY.xlsx'

In [112]:
# sheets_dict = pd.read_excel(filepath, sheet_name=None)
data = pd.read_excel(filepath, sheet_name='Data', header=1)


In [113]:
for col in data.columns:
    print(set(data[col].to_list()))
    
    # financier cols
    # Financier	Financier Public or Private	Financier Type	Domestic/International	Financier Country



{'PTC India', 'Sumitomo Mitsui Trust Holdings', 'FMO', 'Baganuur Power LLC', 'Unknown', 'Axis Bank', 'An Khanh Electricity', 'RPCL-NORINCO Intl Power Limited', 'Vietinbank', 'National Pension Service', 'Huaxi Energy Group', 'HNG Capital', 'BRE Bank', 'IDFC Infrastructure Fund', 'Nippon Life Insurance', 'Banco Centroamericano de Integracion Economica', 'NTPC', 'Verbund', 'Government of Serbia', 'China Development Bank', 'Vivant', 'Adaro Energy', 'African Development Bank', 'Public debentures - Series 1', 'Eren Enerji', 'Malayan Bank', 'Islamic Development Bank', 'Samsung Life Insurance', 'Shizuoka Bank', 'China Energy Group', 'Aboitiz Power Corp', 'Attijariwafa Bank', 'Dena Bank', 'JP Morgan', 'Japan Bank for International Cooperation', 'United Bank of India', 'Mizuho Bank', 'Indico Financial Management & Services', 'Halkbank', 'Woori Bank', 'State Grid Corporation of China', 'Mahanadi Basin Power Limited ', 'Gunma Bank', 'MKCSS Holdings', 'Norinchukin Bank', 'Fatima Group', 'China Inve

In [114]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6556 entries, 0 to 6555
Data columns (total 68 columns):
 #   Column                                                              Non-Null Count  Dtype  
---  ------                                                              --------------  -----  
 0   Financier                                                           6556 non-null   object 
 1   Financier Public or Private                                         6556 non-null   object 
 2   Financier Type                                                      6556 non-null   object 
 3   Domestic/International                                              6556 non-null   object 
 4   Financier Country/Area                                              6548 non-null   object 
 5   Advisor                                                             6556 non-null   object 
 6   Type of Finance                                                     6556 non-null   object 
 7   Equity Partners

In [115]:
## CLEANING ##

data['Financier Country/Area'] = data['Financier Country/Area'].replace('US', 'United States').replace('UAE', 'United Arab Emirates').replace('Democratic Republic of the Congo', 'DR Congo').replace('corporate', 'India').replace('Unknown', 'Not found').replace('','Not found').replace('Pakistan / Iran', 'International').replace('TBD', 'Not found').replace('Chinese', 'China')
data['Financier Country/Area'] = data['Financier Country/Area'].fillna('Not found').apply(lambda x: str(x).strip())

# strip all country names to remove white space after China and Russia

In [116]:

def harmonize_countries(df, countries_dict, test_results_folder):
    df = df.copy()

    region_col = set(df['Region'].to_list())
    results = []
    for region in region_col:
        df_mask = df[df['Region']==region]
        df_mask['country-harmonize-pass'] = df_mask['Country/Area'].apply(lambda x: 'true' if x in countries_dict[region] else f"false because {x}")
        results_len = df_mask[df_mask['country-harmonize-pass'] == 'false']
        results.append((region, len(results_len)))
        print(f'\nWe want this to be 0: {results}\n')
        results_df = pd.DataFrame(results)
        results_df.to_csv(f'{test_results_folder}results.csv')
        
    # df['areas-subnat-sat-display'] = df.apply(lambda row: f"{row['country']}" if row['state/province'] == '' else f"{row['state/province']}, {row['country']}", axis=1)   



def check_countries_official(df,col_country_name, col_wiki):
    df = df.copy()
    problem_units_weird_country = []
    df_country_list = df[col_country_name].unique().tolist()
    df_country_wiki = df[[col_country_name, col_wiki]] # df wiht just these two columns
    for row in df_country_wiki.index:
        if df.loc[row,col_country_name] in full_country_list:
            pass
        else:
            problem_units_weird_country.append((df.loc[row,col_wiki], df.loc[row,col_country_name]))

    df = df.reset_index()
    print(problem_units_weird_country)
    return problem_units_weird_country


In [117]:
data[data['Financier Country/Area']=='International'] # 19 # maybe centroid between the two 
data[data['Financier Country/Area']=='TBD'] # 3
# data[data['Financier Country/Area'] == 'NAN'] # 6

,Financier,Financier Public or Private,Financier Type,Domestic/International,Financier Country/Area,Advisor,Type of Finance,Equity Partnership,Units Included,Financing Status,...,Captive residential use,China Capacity Payment Recipient,CHP,Plant age (years),Capacity factor,Heat rate (Btu per kWh),Emission factor (kg of CO2 per TJ),Annual CO2 (million tonnes / annum),Remaining plant lifetime (years),Lifetime CO2 (million tonnes)


In [118]:
set(data['Financier Country/Area'].to_list())

{'Australia',
 'Austria',
 'Bahamas',
 'Bangladesh',
 'Barbados',
 'Belgium',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Cambodia',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Costa Rica',
 'Czech Republic',
 "Côte d'Ivoire",
 'DR Congo',
 'Dominican Republic',
 'Egypt',
 'El Salvador',
 'France',
 'Germany',
 'Guatemala',
 'Honduras',
 'Hong Kong',
 'India',
 'Indonesia',
 'International',
 'Iran',
 'Ireland',
 'Italy',
 'Japan',
 'Kazakhstan',
 'Kenya',
 'Kuwait',
 'Laos',
 'Luxembourg',
 'Malawi',
 'Malaysia',
 'Mauritius',
 'Mongolia',
 'Morocco',
 'Mozambique',
 'Netherlands',
 'Niger',
 'Nigeria',
 'Not found',
 'Pakistan',
 'Panama',
 'Philippines',
 'Poland',
 'Portugal',
 'Qatar',
 'Romania',
 'Russia',
 'Saudi Arabia',
 'Serbia',
 'Singapore',
 'Slovenia',
 'South Africa',
 'South Korea',
 'Spain',
 'Sri Lanka',
 'Sweden',
 'Switzerland',
 'Taiwan',
 'Tajikistan',
 'Tanzania',
 'Thailand',
 'Togo',
 'Türkiye',
 'United Arab Emirates',
 'United Kingdom',
 'United S

In [119]:
# get stats on which countries have the most capacity and/or money associated using Capacity (MW) 
# deduplicate on unit id so no duplicate capacity by pivotting
# use number of units and This Financier's Unit Share per Transaction to calculate amount money

In [120]:
### CAPACITY ### 

# HOW ATTRIBUTE CAPACITY OF UNIT CORRECTLY TO FINANCER?
# 5 deals, 100 mw each time, use average
# 700 MW
# Orion power station
# 2k a unit or more would be weird! 

In [121]:
### FUNDING ###

# DO I USE THE FINANCING Unit Share per Transaction COLUMN?
#'Total Closed Financing at Plant (All Financiers, All Transactions)': 'dollars',
# # 'All Finance This Plant': 'dollars', 
# switch to Total Closed Financing at Plant (All Financiers, All Transactions)
# Unit Share per Transaction prevents double counting! for dollar amount
 

In [122]:
### COUNTRIES ### 

# Adani should be India for corporate
# Pakistan / Iran 
# nan to unknown? 
# corporate International? if Domestic/International domestic then use centroid, in general use centroid for all financier country


In [123]:
# from previous code
# Get today's date
today_date = date.today()
# Format the date in ISO format
iso_today_date = today_date.isoformat()


In [124]:
# df = data[['Fuel','Technology','Plant name','Wiki name','Plant name (local script)','Owner', 'Parent', 'Capacity (MW)', 'Status', 'Region', 'Country', 'Subnational unit (province, state)', 'Start year','Wiki URL','Latitude','Longitude']]
df = data.rename(columns={'GEM unit/phase ID':'tid','Latitude': 'target_lat', 'Longitude':'target_lng', 'Start year': 'start_year', 'Status': 'status',
                        'Capacity (MW)': 'megawatts', 'Parent': 'parent', 'Owner': 'owner',
                        'Plant name (local)': 'project_loc', 'Plant name': 'project_name', 'Unit name': 'unit',
                        'Subnational unit (province, state)': 'subnational', 'Country/Area':'country', 'Wiki URL':'wiki', 'Region':'region', 'Subregion':'subregion', 'Financier': 'financer',
                        'Financier Type': 'financer_type', 'Type of Finance': 'finance_type', 'Close Year': 'close_year', 'EPC Company': 'epc', "This Financier's Unit Share per Transaction":'dollars', 
                        'Financing Status': 'fin_status', 
                        })

# target is a copy of project_name
df['target'] = df['project_name'] 
# source copy of country 
df['source'] = df['Financier Country/Area']
# get source_iso based on country iso dict 
df['source_iso'] = ''
# sponsor is just empty so not needed maybe?
df['sponsor'] = ''


# target_country_lat,target_country_lng,  copy of target_lat, target_lng
df['target_country_lat'] = df['target_lat']
df['target_country_lng'] = df['target_lng']
df['target_country'] = df['country']
df['dollars'] = df['dollars'].apply(lambda x: (str(x).replace('$', '').replace(',', '').replace('.00','')))
df['dollars'] = df['dollars'].fillna('')

In [125]:
df['unit'] = df['project_name'] + ' ' + df['unit'] 

In [126]:
df['fin_status'] = df['fin_status'].str.lower()
df['finance_type'] = df['finance_type'].str.lower()

In [127]:
set(df['financer_type'].to_list())

{'Government Policy Institution',
 'Government-owned commercial institution',
 'Privately-owned commercial institution',
 'TBD',
 'Unknown'}

In [128]:
df['financer_type'] = df['financer_type'].replace('Government-owned policy institution', 'Governmental policy institution').replace('Joint Venture', 'Joint venture')
df['close_year'] = df['close_year'].apply(lambda x: ((str(x).replace('.0', ''))))

In [129]:
df[['source', 'country', 'target_country']] = df[['source', 'country', 'target_country']].applymap(lambda x: x.strip())


/var/folders/70/9xc3s63n0j9crf3st7fc61cm0000gn/T/ipykernel_71065/1193358027.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[['source', 'country', 'target_country']] = df[['source', 'country', 'target_country']].applymap(lambda x: x.strip())


In [130]:
# df = df.drop_duplicates(['tid'])
print(df['tid'].value_counts(sort=True))
df[['tid', 'country', 'source', 'fin_status', 'megawatts']].to_csv('capacities_same_unit_diff_value.csv')


tid
G100000109986    57
G100000109985    38
G100000105007    33
G100000105008    33
G100000106130    31
                 ..
G100000110529     1
G100000110528     1
G100000110527     1
G100000110526     1
G100001048434     1
Name: count, Length: 2753, dtype: int64


In [131]:
nodupe = df.drop_duplicates(['tid'])


In [132]:
df_capacities_diff_unit = df.drop_duplicates(['tid', 'megawatts'])
print(df_capacities_diff_unit['tid'].value_counts(sort=True))
df_capacities_diff_unit[['tid', 'country', 'source', 'fin_status', 'megawatts']].to_csv('capacities_same_unit_diff_value.csv')


tid
G100001020507    1
G100000115151    1
G100000104801    1
G100000106532    1
G100000108394    1
                ..
G100000104690    1
G100000104691    1
G100000103195    1
G100000103235    1
G100001048434    1
Name: count, Length: 2753, dtype: int64


In [133]:
reppoint = pd.read_csv('/Users/gem-tah/GEM_INFO/GEM_WORK/reppoint_csv_country.csv')
print(reppoint)
print(reppoint.columns)

     Unnamed: 0                                       rep_point  \
0             0   POINT (-69.98430633499993 12.518333912000116)   
1             1     POINT (65.21139794606012 33.92496490550005)   
2             2  POINT (18.873625872703336 -11.940369079499902)   
3             3   POINT (-63.03963163033984 18.218682401500075)   
4             4    POINT (19.84973934280871 60.195574000500045)   
..          ...                                             ...   
258         258    POINT (47.47972127100002 15.796667000000127)   
259         259  POINT (26.127984740653012 -28.480114999999927)   
260         260  POINT (25.450269903236915 -13.178076286999953)   
261         261    POINT (33.828057089285835 35.34988800000016)   
262         262   POINT (29.35411233397813 -19.014528274499867)   

                name  abr  
0              Aruba  ABW  
1        Afghanistan  AFG  
2             Angola  AGO  
3           Anguilla  AIA  
4              Åland  ALA  
..               ...  ...  

In [134]:
centroids = {
     # then replaces all other with arbitrary
    # Latitude: -55.000° Longitude: 71.000°
    'International': [30.0, -40.0],
    'Not found': [-55.0, 71.0], 
    'United Arab Emirates': [23.4241, 53.8478],
    'Kazakhstan': [48.0196, 66.9237],
    'Serbia': [44.0165, 21.0059],
    'Pakistan': [30.3753, 69.3451],
    'Australia': [-25.2744, 133.7751],
    'Dominican Republic': [18.7357, -70.1627],
    'Zambia': [-13.1339, 27.8493],
    'Togo': [8.6195, 0.8248],
    'Taiwan': [23.6978, 120.9605],
    'France': [46.6034, 1.8883],
    'Sri Lanka': [7.8731, 80.7718],
    'Bahamas': [25.0343, -77.3963],
    'Iran': [32.4279, 53.6880],
    'Switzerland': [46.8182, 8.2275],
    'Hong Kong': [22.3193, 114.1694],
    'Nigeria': [9.0820, 8.6753],
    'Brazil': [-14.2350, -51.9253],
    'Botswana': [-22.3285, 24.6849],
    'El Salvador': [13.7942, -88.8965],
    'Romania': [45.9432, 24.9668],
    'Bosnia and Herzegovina': [43.9159, 17.6791],
    'Malawi': [-13.2543, 34.3015],
    'Türkiye': [38.9637, 35.2433],
    'Belgium': [50.5039, 4.4699],
    'Malaysia': [4.2105, 101.9758],
    'Egypt': [26.8206, 30.8025],
    'Mauritius': [-20.3484, 57.5522],
    'Spain': [40.4637, -3.7492],
    'Philippines': [12.8797, 121.7740],
    'Vietnam': [14.0583, 108.2772],
    'Mongolia': [46.8625, 103.8467],
    'Ireland': [53.4129, -8.2439],
    'Tajikistan': [38.8610, 71.2761],
    'Qatar': [25.3548, 51.1839],
    'Kuwait': [29.3117, 47.4818],
    'Barbados': [13.1939, -59.5432],
    'Portugal': [39.3999, -8.2245],
    'Luxembourg': [49.8153, 6.1296],
    "Côte d'Ivoire": [7.5400, -5.5471],
    'Poland': [51.9194, 19.1451],
    'Thailand': [15.8700, 100.9925],
    'Guatemala': [15.7835, -90.2308],
    'Costa Rica': [9.7489, -83.7534],
    'Austria': [47.5162, 14.5501],
    'United Kingdom': [55.3781, -3.4360],
    'Singapore': [1.3521, 103.8198],
    'Morocco': [31.7917, -7.0926],
    'Uzbekistan': [41.3775, 64.5853],
    'Kenya': [-0.0236, 37.9062],
    'Cambodia': [12.5657, 104.9910],
    'United States': [37.0902, -95.7129],
    'Canada': [56.1304, -106.3468],
    'Chile': [-35.6751, -71.5430],
    'Saudi Arabia': [23.8859, 45.0792],
    'Colombia': [4.5709, -74.2973],
    'South Africa': [-30.5595, 22.9375],
    'Panama': [8.5379, -80.7821],
    'Sweden': [60.1282, 18.6435],
    'Japan': [36.2048, 138.2529],
    'Honduras': [15.2000, -86.2419],
    'Indonesia': [-0.7893, 113.9213],
    'China': [35.8617, 104.1954],
    'India': [20.5937, 78.9629],
    'Mozambique': [-18.6657, 35.5296],
    'South Korea': [35.9078, 127.7669],
    'Bangladesh': [23.6850, 90.3563],
    'Germany': [51.1657, 10.4515],
    'Czech Republic': [49.8175, 15.4730],
    'Netherlands': [52.1326, 5.2913],
    'Russia': [61.5240, 105.3188],
    'Slovenia': [46.1512, 14.9955],
    'Italy': [41.8719, 12.5674]
}

In [135]:
reppointdict = {}

for k,v in centroids.items():
    if k in (reppoint['name'].to_list()):
        reppointdict[k] = reppoint[reppoint['name']==k]['rep_point']
    elif k == 'Not found':
        reppointdict['Not found'] = v
    elif k == 'International':
        reppointdict['International'] = v
    elif k == 'Hong Kong':
        reppointdict['Hong Kong'] = v
    elif k == 'Türkiye':
        reppointdict['Türkiye'] = reppoint[reppoint['name']=='Turkey']['rep_point']
    elif k == 'Czech Republic':
        reppointdict['Czech Republic'] = reppoint[reppoint['name']=='Czechia']['rep_point']

    else:
        print(f'{k} from centroids is not in reppoint:\n{reppoint["name"].to_list()}')
    


print(reppointdict['International'])
    

[30.0, -40.0]


In [136]:
# now get all missing into repdict
for country in reppoint['name'].to_list():
    if country in reppointdict.keys():
        pass
    elif country == 'Democratic Republic of the Congo':
        reppointdict['DR Congo'] = reppoint[reppoint['name']==country]['rep_point']
    else:
        reppointdict[country] = reppoint[reppoint['name']==country]['rep_point']


In [137]:
# turn Point geo into reversed format GI map needs

# FROM 'Czechia': POINT (15.534288345200913 49.80386352500011),

# TO 'Czech Republic': [49.8175, 15.4730],
for k,v in reppointdict.items():
    # should be now a list of long lat
    # print(reppointdict[k])
    gi_geo_format = reppointdict[k]
    if type(gi_geo_format) == list:
        pass
    else:
        gi_geo_format = str(reppointdict[k].values[0]).replace('POINT (', '').replace(')', '').split(' ')
        lat = gi_geo_format[1]
        lng = gi_geo_format[0]
        gi_geo_format = [lat, lng]
        # print(f'{k}: {gi_geo_format}')
        reppointdict[k] = gi_geo_format

    # print(f'Final: {reppointdict[k]}')
        
print(reppointdict['International'])
print(reppointdict['Not found'])
# reppointdict['Pakistan / Iran'] = reppointdict['International']
# Democratic Republic of the Congo

[30.0, -40.0]
[-55.0, 71.0]


In [138]:
reppointdict['DR Congo']

['-4.033149956999949', '22.467300676328144']

In [139]:
# assign a reppoint to each country

new_c_fin = df['Financier Country/Area'].to_list()
# print(set(new_c_rec))
# print(set(new_c_fin))
# US UAE
intdf = df[df['Financier Country/Area']=='International']
print(f'LENGTH INT: {len(intdf)}')
# clean countries, nan, corporate, international, 
mask_c_fin_1 = df['Financier Country/Area'] != 'International'
mask_c_fin_2 = df['Financier Country/Area'] != 'corporate'
mask_c_fin_3 = df['Financier Country/Area'] != 'international'
mask_domestic_na = (df['Domestic/International'] == 'Domestic') & (df['Financier Country/Area'].isna())
# print(new_df[mask_domestic_na][['Country', 'Financier Country/Area']]) # correct 
filtered_new_df = df.copy()
mask_notdomestic_na = (filtered_new_df['Domestic/International'] != 'Domestic')
intdf[intdf['Domestic/International'] != 'Domestic']
print(f'LENGTH INT: {len(intdf)}')

# filters out non country values 
# filtered_new_df = df[mask_c_fin_1]
# filtered_new_df = filtered_new_df[mask_c_fin_2]
# filtered_new_df = filtered_new_df[mask_c_fin_3]
# replaces domestic with regular country if Financier Country is na
filtered_new_df.loc[mask_domestic_na, 'Financier Country/Area'] = filtered_new_df.loc[mask_domestic_na, 'country']
# print(filtered_new_df[mask_domestic_na][['Country', 'Financier Country']]) # correct 

# then replaces all other with arbitrary
# Latitude: -75.000° Longitude: 71.000°
all_na_financier_mask = filtered_new_df['Financier Country/Area'].isna()
# df_na = filtered_new_df[all_na_financier_mask]
# df_na = df_na.drop_duplicates(subset='GEM unit/phase ID')
# print((df_na['Capacity (MW)'].sum()))
# df_na.to_csv('df_na_finance.csv')
# print(df_na['Financier Country'])
filtered_new_df.loc[all_na_financier_mask, 'Financier Country/Area'] = 'Not Found' # WIP
# print(filtered_new_df.loc[all_na_financier_mask, 'Financier Country'])
df_cleaned = filtered_new_df.dropna(subset=['Financier Country/Area'])

df_cleaned_ongoing = df_cleaned[df_cleaned['fin_status']=='Closed']
# check that unknown financier country is highest mw
result = df_cleaned_ongoing.groupby('Financier Country/Area')['megawatts'].sum().sort_values(ascending=False)

print(f'this is result: {result}')


# print(len(filtered_new_df))
# print(len(df_cleaned))
# print(len(df_cleaned))
df_cleaned['Financier Country/Area'] = df_cleaned['Financier Country/Area'].str.strip()
new_c_fin = df_cleaned['Financier Country/Area'].to_list()
print(set(new_c_fin))
# print(centroids['Australia'][0])
# make a new column target_lat, target_lng

for row in df_cleaned.index:
    fin_country = df_cleaned.loc[row, 'Financier Country/Area']
    print(fin_country)
    df_cleaned.loc[row,'source_lat'] = reppointdict[f'{fin_country}'][0]
    df_cleaned.loc[row,'source_lng'] = reppointdict[f'{fin_country}'][1]

print(df_cleaned[df_cleaned['Financier Country/Area']=='International'])

LENGTH INT: 20
LENGTH INT: 20
this is result: Series([], Name: megawatts, dtype: float64)
{'Spain', 'Australia', "Côte d'Ivoire", 'France', 'Bosnia and Herzegovina', 'Kenya', 'Kuwait', 'El Salvador', 'Honduras', 'India', 'Hong Kong', 'Nigeria', 'Mozambique', 'Zambia', 'Sweden', 'Austria', 'Poland', 'Singapore', 'Laos', 'Qatar', 'Chile', 'Guatemala', 'South Korea', 'Türkiye', 'Malawi', 'Niger', 'Luxembourg', 'Pakistan', 'Indonesia', 'United Arab Emirates', 'South Africa', 'Not found', 'Bahamas', 'Tanzania', 'Portugal', 'Vietnam', 'Thailand', 'Czech Republic', 'Ireland', 'Russia', 'Egypt', 'Mongolia', 'Barbados', 'Switzerland', 'Germany', 'Iran', 'Zimbabwe', 'Romania', 'Sri Lanka', 'Italy', 'International', 'Cambodia', 'Serbia', 'Uzbekistan', 'United States', 'Malaysia', 'Belgium', 'United Kingdom', 'Mauritius', 'Japan', 'Dominican Republic', 'China', 'Philippines', 'Canada', 'Taiwan', 'Costa Rica', 'Netherlands', 'Saudi Arabia', 'Botswana', 'Bangladesh', 'Morocco', 'Slovenia', 'DR Congo

In [140]:
list_of_cols_gcbft = [
'tid', 'country', 'unit', 'target', 'wiki', 'megawatts', 'status','project_name',
'target_lat', 'target_lng', 'source_lat', 'source_lng', 'parent', 'subnational',
'financer', 'financer_type', 'finance_type', 'close_year', 'epc', 'dollars', 'fin_status',
'source', 'sponsor', 'source_iso', 'target_country_lat', 'target_country_lng', 'target_country', 'Financier Country/Area'
]
df_cleaned = df_cleaned[list_of_cols_gcbft]


In [141]:
df_cleaned_closed = df_cleaned.drop_duplicates('tid')
df_cleaned_closed = df_cleaned_closed[df_cleaned_closed['fin_status']=='closed'][['megawatts', 'country', 'target_country', 'source', 'tid', 'project_name', 'unit', 'close_year']]

print(df_cleaned_closed['megawatts'].sum())

print(set(df_cleaned_closed['close_year'].to_list()))
# print(df_cleaned_closed.dtypes)
valid_close_years = ['2002','2003','2004','2005','2006','2007','2008','2009','2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

df_cleaned_closed = df_cleaned_closed[df_cleaned_closed['close_year'].isin(valid_close_years)]
# print(df_cleaned_closed['megawatts'].sum())
# df_cleaned_closed_groupby = df_cleaned_closed.groupby('country')
# print(df_cleaned_closed_groupby)
df_cleaned_closed.to_csv(f'closed_by_country_recipient_{iso_today_date}.csv')
# /Users/gem-tah/GEM_INFO/GEM_WORK/greeninfo-maps/coal-finance-main
# output_file = f'/Users/gem-tah/GEM_INFO/GEM_WORK/greeninfo-maps/coal-finance-main/data/gcpft_map_2025_dec_{iso_today_date}.csv'
output_file = f'data/gcpft_map_2025_dec_{iso_today_date}.csv'
# df_cleaned[['source', 'country', 'target_country']] = df_cleaned[['source', 'country', 'target_country']].applymap(lambda x: x.strip())
# Ensure all latitude and longitude columns are converted to float
lat_lng_columns = ['source_lat', 'source_lng', 'target_lat', 'target_lng', 'target_country_lat', 'target_country_lng']
for col in lat_lng_columns:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Drop rows where any of the latitude or longitude columns are NaN or invalid
df_cleaned = df_cleaned.dropna(subset=lat_lng_columns)

# Ensure all latitude and longitude values are within valid ranges
for col in ['source_lat', 'target_lat', 'target_country_lat']:
    df_cleaned = df_cleaned[(df_cleaned[col] >= -90) & (df_cleaned[col] <= 90)]
for col in ['source_lng', 'target_lng', 'target_country_lng']:
    df_cleaned = df_cleaned[(df_cleaned[col] >= -180) & (df_cleaned[col] <= 180)]

for col in df.columns:
    # strip all values
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    
    

        
# TODO 
# deal with finance type
# bond, gov grant, refi bond, refi loan
# deal with institution type
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Tbd", "Not found")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Unknown", "Not found").fillna("Not found").apply(lambda x: x.capitalize())
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Private-owned commercial institution", "Private-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Private commercial institution", "Private-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Government-owned commercial institution", "Government-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Government policy institution", "Governmental policy")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Government policy insitution", "Governmental policy")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Government policy", "Governmental policy")

df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Private commercial institution", "Private-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Commercial", "Private-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Private commercial institution", "Private-owned commercial")
df_cleaned['financer_type'] = df_cleaned['financer_type'].replace("Governmental policy institution", "Governmental policy")



# 'Tbd', 'Government policy', 'Private-owned commercial', 'Not found'

# Private commercial institution 
# Governmental policy institution
# Government policy institution
# Government policy insitution
# Private commercial institution
# Commercial 
print(set(df_cleaned['financer_type'].to_list()))
print(set(df_cleaned['fin_status'].to_list()))
# {'Government-owned commercial', 'Tbd', 'Government policy', 
# 'Private-owned commercial', 'Not found'}
# {'unknown', 'stopped', 'closed', 'financing'}
# {'Bond', 'Equity', 'Loan', 'Refinancing loan', 
# 'Insurance', 'Construction finance', 'Retirement refinancing loan', 
# 'Government grant', 'Refinancing bond', 'Not found'}
# input(f'check above')

df_cleaned['fin_status'] = df_cleaned['fin_status'].fillna('unknown')
df_cleaned['fin_status'] = df_cleaned['fin_status'].apply(lambda x: x.strip())

print(set(df_cleaned['fin_status'].to_list()))

print(set(df_cleaned['finance_type'].to_list()))

df_cleaned['finance_type'] = df_cleaned['finance_type'].replace("Unknown", "Not found").fillna("Not found").apply(lambda x: x.strip().capitalize())
# Grant
df_cleaned['finance_type'] = df_cleaned['finance_type'].replace("Grant", "Government grant")
# Tbd
df_cleaned['finance_type'] = df_cleaned['finance_type'].replace("Tbd", "Not found")
# Construction finance
print(set(df_cleaned['finance_type'].to_list()))

df_cleaned['financer'] = df_cleaned['financer'].fillna("Not found").replace("TBD", "Not found").replace("TBD ", "Not found").replace("Unknown", "Not found")
# Construction finance
print(set(df_cleaned['financer'].to_list()))
# joint venture
# finance destination
# show only domestic WORKS 
# change unknown to not found and remove nan from institution type values
# change nan financer to not found and unknonw to not found

# how to change drop down in interface
# DR congo where it go? Zimbabwe amount correct? 

# can the same financier and unit id and megawatts happen? 


# fin type issues
# retirement refinancing loan,
# construction finance
# Insurance
# loan
print(set(df_cleaned['finance_type'].to_list()))
# {'Bond', 'Equity', 'Loan', 'Refinancing loan', 'Insurance', 'Construction finance', 'Retirement refinancing loan', 'Government grant', 'Refinancing bond', 'Not found'}

# institution type
# private owned commercial
# governmental policy
# joint venture
# not found


df_cleaned.to_csv(output_file, encoding='utf-16', index=False)
print(f'done {output_file}')


303423.7
{'nan', '2011', '2025', '2002', '2010', '2018', 'Unknown', '2017', '2016', '2013', '2023', '2024', '2009', '2012', '2008', '2015', '2014', '2006', '2022', '2020', '2007', '2003', '2005', '2019', '2021'}
{'Government-owned commercial', 'Tbd', 'Not found', 'Privately-owned commercial institution', 'Governmental policy'}
{'unknown', nan, 'stopped ', 'financing', 'closed', 'stopped', 'unknown ', 'financing '}
{'closed', 'stopped', 'unknown', 'financing'}
{'unknown', 'refinancing loan', 'loan', 'tbd', 'retirement refinancing loan', 'bond', 'equity', 'refinancing bond', 'construction finance', 'grant', 'insurance'}
{'Bond', 'Loan', 'Retirement refinancing loan', 'Insurance', 'Not found', 'Construction finance', 'Equity', 'Government grant', 'Refinancing bond', 'Refinancing loan', 'Unknown'}
{'PTC India', 'Sumitomo Mitsui Trust Holdings', 'FMO', 'Baganuur Power LLC', 'Axis Bank', 'An Khanh Electricity', 'RPCL-NORINCO Intl Power Limited', 'Vietinbank', 'National Pension Service', 'Hua

In [142]:
# list_of_cols_gcbft = [
# 'tid', 'country', 'unit', 'target', 'project_name', 'wiki', 'megawatts', 'status',
# 'target_lat', 'target_lng', 'source_lat', 'source_lng', 'parent', 'subnational',
# 'financer', 'financer_type', 'finance_type', 'close_year', 'epc', 'dollars', 'fin_status',
# 'source', 'sponsor', 'source_iso', 'target_country_lat', 'target_country_lng', 'target_country'
# ]
# # 'sponsor', 'source_iso', 'target_country_lat', 'target_country_lng'
# df_cleaned = df_cleaned[list_of_cols_gcbft]
# df_cleaned = df_cleaned.dropna(subset=['source_lat', 'source_lng', 'target_lat', 'target_lng', 'target_country_lat', 'target_country_lng'])


In [143]:
set(df_cleaned['target_country_lat'].isna().to_list())
# set(df_cleaned['target_country_lng'].isna().to_list())

{False}